# Honest-GAP prediction identity

Use `honest_gap` only when an HonestForest exposes structure and estimation roles.


All examples use local synthetic data, fixed seeds, and a reduced smoke
configuration by default. `original`, `kerf`, `path`, `random_height`, and the
separately named isolation `collision` relation are symmetric affinities.
`rfgap`, `honest_gap`, and conditional weights are directed query-to-training
weights; downstream visualization, clustering, or attribution must state any
symmetrization. Prediction identity is claimed only for unsymmetrized `rfgap`
or `honest_gap` under their declared bootstrap/honesty contracts.

In [1]:
SMOKE = True
RANDOM_STATE = 0
N_SAMPLES = 80 if SMOKE else 800
N_ESTIMATORS = 20 if SMOKE else 300

In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=N_SAMPLES, n_features=8, n_informative=5,
    n_redundant=1, random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
)

In [3]:
import inspect
import warnings
from forestgap import Proximity, prediction_identity

try:
    from treeple import HonestForestClassifier
except ImportError:
    HonestForestClassifier = None

if HonestForestClassifier is None:
    result = {"status": "skipped", "reason": "install forestgap[treeple]"}
else:
    signature = inspect.signature(HonestForestClassifier.__init__)
    kwargs = {"n_estimators": N_ESTIMATORS, "random_state": RANDOM_STATE}
    for name, value in (("bootstrap", True), ("honest_fraction", 0.5), ("max_depth", 5), ("n_jobs", 1), ("kernel_method", True)):
        if name in signature.parameters:
            kwargs[name] = value
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        forest = HonestForestClassifier(**kwargs).fit(X_train, y_train)
        proximity = Proximity(forest, relation="honest_gap").attach(X_train, y_train)
    reconstructed = prediction_identity(proximity, X_test)
    result = {
        "status": "ok",
        "relation": proximity.relation,
        "max_abs_error": float(np.max(np.abs(reconstructed - forest.predict_proba(X_test)))),
    }
result

{'status': 'ok',
 'relation': 'honest_gap',
 'max_abs_error': 2.220446049250313e-16}